In [1]:
# ==========================================================
# Install all required libraries for the RAG project
# Run this cell only once in Google Colab
# ==========================================================

!pip -q install langchain
!pip -q install langchain-community
!pip -q install langchain-core
!pip -q install langchain-huggingface

!pip -q install sentence-transformers

!pip -q install faiss-cpu

!pip -q install pypdf
!pip -q install docx2txt

!pip -q install pandas

# ==========================================================
# Install Google Gemini
# ==========================================================

!pip install -q langchain-google-genai

**Importing Libraries**

In [30]:
# ==========================================================
# Import built-in library
# ==========================================================

# Used for working with file paths
import os


# ==========================================================
# Import document loaders
# ==========================================================

# Load PDF files
from langchain_community.document_loaders import PyPDFLoader

# Load DOCX files
from langchain_community.document_loaders import Docx2txtLoader

# Load TXT files
from langchain_community.document_loaders import TextLoader

# Load CSV files
from langchain_community.document_loaders import CSVLoader


# ==========================================================
# Data analysis library (used for CSV inspection if needed)
# ==========================================================

import pandas as pd

# ==========================================================
# Import Regular Expressions Library
# Used for cleaning text using pattern matching
# ==========================================================

import re

# ==========================================================
# Import Text Splitter
# Used to split long documents into smaller chunks
# ==========================================================

from langchain_text_splitters import RecursiveCharacterTextSplitter

# ==========================================================
# Import HuggingFace Embeddings
# ==========================================================

from langchain_huggingface import HuggingFaceEmbeddings

# ==========================================================
# Import FAISS Vector Store
# Used to store embeddings for similarity search
# ==========================================================

from langchain_community.vectorstores import FAISS

# Prompt Template
from langchain_core.prompts import ChatPromptTemplate

# ==========================================================
# Import Google Gemini
# ==========================================================

from langchain_google_genai import ChatGoogleGenerativeAI

**Define Project Paths**

In [3]:
# ==========================================================
# Define the main project directory
# ==========================================================

# Main project folder
PROJECT_PATH = "/content"

# Data folder (Material)
DATA_PATH = PROJECT_PATH

**Display Files**

In [4]:
# ==========================================================
# Display all files inside the data folder
# ==========================================================

print("=" * 60)
print("Available Course Materials")
print("=" * 60)

# Loop through every file inside the data folder
for file_name in os.listdir(DATA_PATH):

    print(file_name)

Available Course Materials
.config
Machine-Learning-.docx
Deep Learning.pdf
Course_Information(1).csv
faiss_index
Generative AI Fundementals.txt
drive
.ipynb_checkpoints
sample_data


**Load Documents**

In [5]:
# ==========================================================
# Load Machine Learning PDF
# ==========================================================

ml_loader = Docx2txtLoader(
    os.path.join(DATA_PATH, "Machine-Learning-.docx")
)

ml_documents = ml_loader.load()


# ==========================================================
# Load Deep Learning PDF
# ==========================================================

dl_loader = PyPDFLoader(
    os.path.join(DATA_PATH, "Deep Learning.pdf")
)

dl_documents = dl_loader.load()


# ==========================================================
# Load Generative AI TXT
# ==========================================================

genai_loader = TextLoader(
    os.path.join(DATA_PATH, "Generative AI Fundementals.txt")
)

genai_documents = genai_loader.load()


# ==========================================================
# Load Course Information CSV
# ==========================================================

course_loader = CSVLoader(
    os.path.join(DATA_PATH, "Course_Information(1).csv")
)

course_documents = course_loader.load()

**Merge All Documents**

In [6]:
# ==========================================================
# Combine all loaded documents into one list
# ==========================================================

documents = (
    ml_documents
    + dl_documents
    + genai_documents
    + course_documents
)

In [7]:
# ==========================================================
# Display loading summary
# ==========================================================

print("=" * 60)
print("Loading Summary")
print("=" * 60)

print(f"Machine Learning Docs : {len(ml_documents)}")
print(f"Deep Learning Pages    : {len(dl_documents)}")
print(f"Generative AI Docs     : {len(genai_documents)}")
print(f"Course Rows            : {len(course_documents)}")
print("-" * 60)
print(f"Total Documents        : {len(documents)}")

Loading Summary
Machine Learning Docs : 1
Deep Learning Pages    : 83
Generative AI Docs     : 1
Course Rows            : 9
------------------------------------------------------------
Total Documents        : 94


**Text Cleaning**

In [8]:
# ==========================================================
# Clean Text Function
# Removes unnecessary content before Chunking
# ==========================================================

import re


def clean_text(documents):

    # List to store cleaned documents
    cleaned_documents = []

    # Loop through every loaded document
    for document in documents:

        # Get current text
        text = document.page_content

        # Get current file name
        file_name = document.metadata.get("source", "").lower()

        # ==========================================================
        # Machine Learning Cleaning --> remove table of contents
        # ==========================================================

        if "machine" in file_name:

            start_phrase = "Machine learning is a branch of artificial intelligence"

            if start_phrase in text:

                text = text[text.find(start_phrase):]

        # ==========================================================
        # Deep Learning Cleaning
        # Remove Cover + Objectives + Index
        # ==========================================================

        elif "deep" in file_name:

            page_number = document.metadata.get("page", 0)

            # Skip first 3 pages completely
            if page_number < 3:
                continue

            # First real page
            if page_number == 3:

                start_phrase = "Deep Learning is a subset of Machine Learning"

                if start_phrase in text:

                    text = text[text.find(start_phrase):]

        # ==========================================================
        # General Cleaning
        # ==========================================================

        # Remove URLs
        text = re.sub(r'https?://\S+|www\.\S+', ' ', text)

        # Remove Emails
        text = re.sub(r'\S+@\S+', ' ', text)

        # Remove HTML Tags
        text = re.sub(r'<.*?>', ' ', text)

        # Remove Weird Symbols
        text = re.sub(r'[^\w\s.,!?():/\-]', ' ', text)

        # Remove Extra Spaces
        text = re.sub(r'\s+', ' ', text)

        # Remove Leading & Trailing Spaces
        text = text.strip()

        # Update document text
        document.page_content = text

        # Save cleaned document
        cleaned_documents.append(document)

    return cleaned_documents

In [9]:
# ==========================================================
# Apply Cleaning
# ==========================================================

cleaned_documents = clean_text(documents)

In [10]:
# ==========================================================
# Preview Cleaned Documents
# ==========================================================

print(cleaned_documents[0].page_content[:500])
print("="*80)
print(cleaned_documents[1].page_content[:500])

Machine learning is a branch of artificial intelligence that focuses on creating algorithms that can learn from data and make predictions or decisions based on that data. The goal of machine learning is to enable machines to learn from experience, so that they can improve their performance over time. In this chapter, we will explore the fundamentals of machine learning, including its definition, history, and key concepts. Definition and History of Machine Learning Machine learning has its roots 
Deep Learning is a subset of Machine Learning (ML) that uses artificial neural networks with many layers to learn patterns from large amounts of data. It is inspired by the structure and function of the human brain. Deep learning is especially powerful in handling unstructured data like images, audio, video, and text. Deep learning models automatically learn features from raw data by passing it through multiple layers (input hidden layers output). Each layer learns different levels of abstracti

**Text Chunking**



*   **Create Text Splitter**



In [11]:
# ==========================================================
# Create Recursive Text Splitter
# ==========================================================

text_splitter = RecursiveCharacterTextSplitter(

    # Maximum number of characters in each chunk
    chunk_size=1000,

    # Number of overlapping characters between chunks
    chunk_overlap=200,

    # Function used to count characters
    length_function=len,

    # Try to split using larger separators first
    separators=[
        "\n\n",   # Paragraph
        "\n",     # New line
        ".",      # Sentence
        " ",      # Space
        ""        # Character
    ]
)



*   **Split Documents**



In [12]:
# ==========================================================
# Split All Documents into Chunks
# ==========================================================

chunks = text_splitter.split_documents(documents)

In [13]:
# ==========================================================
# Display Chunking Summary
# ==========================================================

print("=" * 60)
print("Chunking Summary")
print("=" * 60)

print(f"Original Documents : {len(documents)}")
print(f"Generated Chunks   : {len(chunks)}")

Chunking Summary
Original Documents : 94
Generated Chunks   : 434


In [14]:
# ==========================================================
# Display the First Three Chunks in Machine Learning
# ==========================================================

for index, chunk in enumerate(chunks[:3]):

    print("=" * 60)
    print(f"Chunk {index + 1}")
    print("=" * 60)

    print(chunk.page_content[:500])

    print("\n")

Chunk 1
Machine learning is a branch of artificial intelligence that focuses on creating algorithms that can learn from data and make predictions or decisions based on that data. The goal of machine learning is to enable machines to learn from experience, so that they can improve their performance over time. In this chapter, we will explore the fundamentals of machine learning, including its definition, history, and key concepts. Definition and History of Machine Learning Machine learning has its roots 


Chunk 2
. Today, machine learning has become one of the most exciting and rapidly growing fields in computer science, with applications in a wide range of industries such as healthcare, finance, and transportation. As the amount of data being generated continues to grow exponentially, machine learning is increasingly being used to help make sense of this data, uncover patterns, and make predictions. Key Concepts in Machine Learning In the field of machine learning, it is essential to 

In [15]:
# ==========================================================
# Display Metadata of the First Chunk
# ==========================================================

print("=" * 60)
print("Chunk Metadata")
print("=" * 60)

print(chunks[0].metadata)

Chunk Metadata
{'source': '/content/Machine-Learning-.docx'}


In [16]:
# ==========================================================
# Display the Size of the First Five Chunks
# ==========================================================

for index, chunk in enumerate(chunks[:5]):

    print(f"Chunk {index + 1}")

    print(f"Characters : {len(chunk.page_content)}")

    print("-" * 50)

Chunk 1
Characters : 995
--------------------------------------------------
Chunk 2
Characters : 998
--------------------------------------------------
Chunk 3
Characters : 966
--------------------------------------------------
Chunk 4
Characters : 737
--------------------------------------------------
Chunk 5
Characters : 863
--------------------------------------------------


**Text Embedding**



*   **Load Embedding Model**



In [17]:
# ==========================================================
# Load Embedding Model
# ==========================================================

embedding_model = HuggingFaceEmbeddings(

    # Name of the embedding model from Hugging Face
    model_name="sentence-transformers/all-MiniLM-L6-v2",

    # Model configuration
    model_kwargs={
        "device": "cuda"
    },

    # Encoding configuration
    encode_kwargs={
        "normalize_embeddings": True
    }
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [18]:
# ==========================================================
# Test the Embedding Model
# ==========================================================

sample_embedding = embedding_model.embed_query(
    "What is Machine Learning?"
)

print("=" * 60)
print("Embedding Test")
print("=" * 60)

print(f"Vector Length : {len(sample_embedding)}")

print("\nFirst 10 Values:")

print(sample_embedding[:10])

Embedding Test
Vector Length : 384

First 10 Values:
[-0.019954577088356018, 0.009877998381853104, 0.010249645449221134, 0.029553700238466263, 0.02718641236424446, -0.0192965529859066, -0.02412964217364788, -0.0377357117831707, -0.041054196655750275, -0.0014750208938494325]


**Create Vector Database (FAISS)**

In [19]:
# ==========================================================
# Create FAISS Vector Database
# ==========================================================

vector_database = FAISS.from_documents(

    # List of document chunks
    documents=chunks,

    # Embedding model
    embedding=embedding_model

)

In [20]:
# ==========================================================
# Vector Database Summary
# ==========================================================

print("=" * 60)
print("Vector Database Created Successfully")
print("=" * 60)

print(f"Total Chunks Stored : {len(chunks)}")

Vector Database Created Successfully
Total Chunks Stored : 434


**Save Vector DB Model**

In [21]:
# ==========================================================
# Save FAISS Vector Database
# ==========================================================

FAISS_PATH = "/content/faiss_index"
vector_database.save_local(FAISS_PATH)

print("FAISS Index Saved Successfully!")

**Build the Retriever**

In [24]:
# ==========================================================
# Create Retriever
# Responsible for retrieving the most relevant chunks
# ==========================================================

# change FAISS to retriever
retriever = vector_database.as_retriever(

    # Search type
    search_type="similarity",

    # Search configuration
    search_kwargs={

        # Return the top 3 most similar chunks
        "k": 3

    }

)



*   **Test Retriever**



In [25]:
# ==========================================================
# Test Retriever
# ==========================================================

query = "What is Machine Learning?"

retrieved_chunks = retriever.invoke(query)


# ==========================================================
# Display Retrieved Chunks
# ==========================================================

print("=" * 60)
print("Retrieved Chunks")
print("=" * 60)

for index, chunk in enumerate(retrieved_chunks):

    print(f"\nChunk {index + 1}")
    print("-" * 60)
    print(chunk.page_content[:500])
    print("\nMetadata:")
    print(chunk.metadata)

Retrieved Chunks

Chunk 1
------------------------------------------------------------
Machine learning is a branch of artificial intelligence that focuses on creating algorithms that can learn from data and make predictions or decisions based on that data. The goal of machine learning is to enable machines to learn from experience, so that they can improve their performance over time. In this chapter, we will explore the fundamentals of machine learning, including its definition, history, and key concepts. Definition and History of Machine Learning Machine learning has its roots 

Metadata:
{'source': '/content/Machine-Learning-.docx'}

Chunk 2
------------------------------------------------------------
. Today, machine learning has become one of the most exciting and rapidly growing fields in computer science, with applications in a wide range of industries such as healthcare, finance, and transportation. As the amount of data being generated continues to grow exponentially, machine

**Prompt Template**

In [37]:
prompt = ChatPromptTemplate.from_template("""
You are an AI assistant that answers students' questions.

Use ONLY the provided context.

If the answer is not found in the context,
reply exactly:

"I don't know based on the provided course materials."

Give the answer in a clear and well-formatted way.

If there are multiple definitions or points:
- Use bullet points.
- Put each point on a new line.
- Keep the answer concise.

----------------------------

Context:
{context}

----------------------------

Question:
{question}

----------------------------

Answer:
""")



*   **Test Prompt**



In [38]:
# ==========================================================
# Test Prompt
# ==========================================================

query = "What is Machine Learning?"

retrieved_docs = retriever.invoke(query)

context = "\n\n".join(
    doc.page_content for doc in retrieved_docs
)

final_prompt = prompt.invoke(
    {
        "context": context,
        "question": query
    }
)

print(final_prompt)

messages=[HumanMessage(content='\nYou are an AI assistant that answers students\' questions.\n\nUse ONLY the provided context.\n\nIf the answer is not found in the context,\nreply exactly:\n\n"I don\'t know based on the provided course materials."\n\nGive the answer in a clear and well-formatted way.\n\nIf there are multiple definitions or points:\n- Use bullet points.\n- Put each point on a new line.\n- Keep the answer concise.\n\n----------------------------\n\nContext:\nMachine learning is a branch of artificial intelligence that focuses on creating algorithms that can learn from data and make predictions or decisions based on that data. The goal of machine learning is to enable machines to learn from experience, so that they can improve their performance over time. In this chapter, we will explore the fundamentals of machine learning, including its definition, history, and key concepts. Definition and History of Machine Learning Machine learning has its roots in the field of statis

**Call LLM**

In [ ]:


# ==========================================================
# Create Gemini Model
# ==========================================================

from langchain_google_genai import ChatGoogleGenerativeAI


llm = ChatGoogleGenerativeAI(

    # Gemini model
    model="gemini-3.5-flash",

    google_api_key=os.getenv("GOOGLE_API_KEY"),

    # Lower temperature = more factual answers
    temperature=0
)

/usr/local/lib/python3.12/dist-packages/langchain_core/language_models/chat_models.py:431: UserWarning: Unrecognized keys in model profile: ['reasoning_effort_default', 'reasoning_effort_levels']. This may indicate a version mismatch between langchain-core and your provider package. Consider upgrading langchain-core.
  _warn_unknown_profile_keys(self.profile)


In [42]:
response = llm.invoke(final_prompt)

print(response.content[0]["text"]) # we form with this way because (response.content) is "list" not "string"

Based on the provided course materials, machine learning is:

* A branch of artificial intelligence that focuses on creating algorithms that can learn from data and make predictions or decisions based on that data.
* A tool that enables computers to learn from data and make predictions or decisions without being explicitly programmed.
* An approach with the goal of enabling machines to learn from experience so they can improve their performance over time.


**Build the RAG Chain**



*   we build RAG Chain for for grouping all stages together and calling them once instead of calling them as individual



In [43]:
# ==========================================================
# Import Required LangChain Components
# ==========================================================
# RunnablePassthrough:
# Passes the user's question directly to the prompt.
#
# StrOutputParser:
# Converts the LLM output into a plain string.
# ==========================================================

from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [44]:
# ==========================================================
# Convert Retrieved Documents into One Context String
# ==========================================================
# The retriever returns a list of Document objects.
# We join their page_content into one text block.
# ==========================================================

def format_docs(docs):

    return "\n\n".join(doc.page_content for doc in docs)


*   **RAG Chain Archticture**




In [45]:
# ==========================================================
# Build the Complete RAG Chain
# ==========================================================
#
# User Question
#        │
#        ▼
# Retriever
#        │
# Retrieved Chunks
#        │
# format_docs()
#        │
# Prompt Template
#        │
# Gemini
#        │
# Final Answer
#
# ==========================================================

rag_chain = (

    {
        "context": retriever | format_docs,

        "question": RunnablePassthrough()

    }

    | prompt

    | llm

    | StrOutputParser()

)



*   **Testing**



In [46]:
# ==========================================================
# Ask a Question
# ==========================================================

question = "What is Machine Learning?"
answer = rag_chain.invoke(question)

# ==========================================================
# Display Answer
# ==========================================================

print("=" * 70)
print("Answer")
print("=" * 70)

print(answer)

Answer
Based on the provided course materials, machine learning is:

* A branch of artificial intelligence that focuses on creating algorithms that can learn from data and make predictions or decisions based on that data.
* A tool that enables computers to learn from data and make predictions or decisions without being explicitly programmed.
* An approach with the goal of enabling machines to learn from experience so they can improve their performance over time.
